In [1]:
import numpy as np
import pandas as pd
# from lightgbm import LGBMRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

"""
Есть набор признаков, которые вычисляются независимо от эксперимента. ?
Используя эти признаки, нужно разбить объекты на страты так,
чтобы дисперсия стратифицированного среднего была минимальна и доля каждой страты была не менее 5% от всех данныx.

Данные разбиты на 2 части.
Решение будет проверяться на второй части данных.
Значения в столбцах x1, ..., x10 — признаки, которые можно использовать для вычисления страт.
Значения в столбце y — измерения, по которым будет вычисляться целевая метрика эксперимента.

Дисперсия должна не превышать 50000.
"""

def get_strats(df_features):
    """Возвращает страты объектов.
    
    :param df_features (pd.DataFrame): таблица с признаками x1, ..., x10
    :return (list | np.array | pd.Series): список страт объектов размера len(df).
    """
    # df_train = df.iloc[:len(df) // 2].copy()
    # df_test = df.iloc[len(df) // 2:].copy()
    df_train = df_features.copy()
    df_test = df_features.copy()
    
    # Initialize the DecisionTreeRegressor
    # model = LGBMRegressor(num_leaves=3)
    regressor = DecisionTreeRegressor(random_state=42)
    
    # Train the model
    # model.fit(df_train[feature_names].values, df_train['y'].values)
    feature_names = [f'x{i}' for i in range(1, 11)]
    regressor.fit(df_train[feature_names], df_train['y'])
    
    # Make predictions
    # predict_test = model.predict(df_test[feature_names].values)
    predict_test = regressor.predict(df_test[feature_names])
    
    n_strat = 10
    quantiles = np.quantile(predict_test, np.linspace(0, 1 - 1 / n_strat, n_strat))
    df_test['strata'] = [np.sum(predict >= quantiles) for predict in predict_test]
    return df_test.strata


In [2]:
# df = pd.read_csv('chapter_08_stratification_task_data_public.csv')
# feature_min_var = get_feature_with_min_stratified_variance(df)
# feature_min_var = 'x2'
# feature_values_to_unite = get_features_to_unite(df, feature_min_var, 'y')
# df_united = unite_feature_values(df, feature_min_var, feature_values_to_unite)
# params = calculate_strata_params(df_united)
# print(params)


df = pd.read_csv('data/chapter_08_stratification_task_data_public.csv')

strata = get_strats(df)

print(strata)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000832 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 800
[LightGBM] [Info] Number of data points in the train set: 10000, number of used features: 10
[LightGBM] [Info] Start training from score 1367.874000
0        8
1        8
2        6
3        2
4        1
        ..
9995    10
9996     5
9997     9
9998     2
9999     3
Name: strata, Length: 10000, dtype: int64


/home/pavel/.conda/envs/jupyter-env/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/pavel/.conda/envs/jupyter-env/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
